# **Installing and importing the required libraries**

In [ ]:
!pip install tqdm rarfile nibabel
import os, nibabel as nib, numpy as np, rarfile
from PIL import Image
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Data Loading**

In [ ]:
RAR_PATH = '/content/drive/MyDrive/ADNI_MRI_Dataset.rar'
EXTRACT_PATH = '/content/temp_nifti'
OUTPUT_ROOT = '/content/drive/MyDrive/ADNI_MultiView_Dataset'

In [ ]:
# EXTRACTION STEP (If not already extracted)
if not os.path.exists(EXTRACT_PATH):
    print("Extracting RAR file... please wait.")
    with rarfile.RarFile(RAR_PATH) as rf:
        rf.extractall(EXTRACT_PATH)

# **Extracts 3 slice from each patient (axial, coronal, sagittal)**

In [ ]:
base_data_path = None
for root, dirs, files in os.walk(EXTRACT_PATH):
    if 'train' in dirs:
        base_data_path = root
        break

if not base_data_path:
    print("CRITICAL ERROR: Could not find 'train' folder. Check RAR structure.")
else:
    print(f"Found data at: {base_data_path}")

    splits = ['train', 'test', 'val']
    labels = ['CN', 'AD', 'MCI']

    for split in splits:
        for label in labels:
            src = os.path.join(base_data_path, split, label)
            dst = os.path.join(OUTPUT_ROOT, split, label)
            os.makedirs(dst, exist_ok=True)

            if os.path.exists(src):
                files = [f for f in os.listdir(src) if f.endswith('.nii.gz')]
                if len(files) == 0:
                    continue

                for f in tqdm(files, desc=f"Converting {split}/{label}"):
                    try:
                        # Load NIfTI
                        img = nib.load(os.path.join(src, f))
                        data = img.get_fdata()
                        patient_id = f.replace('.nii.gz', '')

                        # --- MULTI-VIEW EXTRACTION WITH ORIENTATION FIX ---
                        # We apply .T to get X/Y aligned, then flipud to put the head at the top
                        slices = {
                            'axial': np.flipud(data[:, :, data.shape[2] // 2].T),
                            'coronal': np.flipud(data[:, data.shape[1] // 2, :].T),
                            'sagittal': np.flipud(data[data.shape[0] // 2, :, :].T)
                        }

                        for view_name, img_slice in slices.items():
                            # Independent Normalization (0-255)
                            d_min, d_max = np.min(img_slice), np.max(img_slice)
                            if d_max > d_min:
                                img_slice = ((img_slice - d_min) / (d_max - d_min) * 255).astype(np.uint8)
                            else:
                                img_slice = np.zeros(img_slice.shape, dtype=np.uint8)

                            # SAVE STEP:
                            output_name = f"{patient_id}_{view_name}.png"
                            Image.fromarray(img_slice).convert('L').save(os.path.join(dst, output_name))

                    except Exception as e:
                        print(f"Skipping {f}: {e}")

print(f"\nSuccess! Multi-view images saved in: {OUTPUT_ROOT}")

Found data at: /content/temp_nifti/ADNI_MRI Dataset


Converting val/MCI: 100%|██████████| 115/115 [00:29<00:00,  3.85it/s]


Success! Multi-view images saved in: /content/drive/MyDrive/ADNI_MultiView_Dataset
